# 181 — Logit Lens Surgery

Surgical logit lens: apply logit lens at specific layers, compare between
layers, measure intervention effects, track component contributions,
and follow prediction trajectories.

## Why This Matters

Intervention logit lens surgery provides tools for controlled modifications of model internals. Causal interventions are the gold standard for mechanistic claims — they establish not just what correlates with behavior, but what actually causes it.

**Key references:**
- [Turner et al. (2023) "Activation Addition"](https://arxiv.org/abs/2308.10248) — Steering model behavior by adding vectors to activations
- [Wang et al. (2023) "Interpretability in the Wild: IOI"](https://arxiv.org/abs/2211.00593) — Detailed circuit analysis of indirect object identification

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.logit_lens_surgery import (
    logit_lens_at_layer,
    logit_lens_diff,
    logit_lens_intervention,
    component_logit_lens_effect,
    logit_lens_trajectory,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.arange(15)

In [ ]:
for l in range(4):
    result = logit_lens_at_layer(model, tokens, layer=l)
    preds = [p['top_token'] for p in result['per_position'][-3:]]
    print(f"Layer {l}: conf={result['mean_confidence']:.3f}  "
          f"entropy={result['mean_entropy']:.3f}  last_preds={preds}")

In [ ]:
result = logit_lens_diff(model, tokens, layer_a=0, layer_b=3)
print(f"L0 -> L3: {result['n_changed']}/{len(result['per_position'])} changed  "
      f"mean_KL={result['mean_kl']:.4f}\n")
for p in result['per_position']:
    changed = ' CHANGED' if p['prediction_changed'] else ''
    print(f"  pos {p['position']}: tok{p['top_token_a']}->tok{p['top_token_b']}  "
          f"KL={p['kl_divergence']:.4f}{changed}")

In [ ]:
def zero_hook(x, name):
    return jnp.zeros_like(x)

result = logit_lens_intervention(model, tokens, layer=3,
                                  hook_name='blocks.1.hook_attn_out',
                                  hook_fn=zero_hook)
print(f"Zeroing L1 attn, measuring at L3:")
print(f"Changed: {result['n_changed']}  mean_max_change={result['mean_max_change']:.4f}\n")
for p in result['per_position']:
    status = ' CHANGED' if p['changed'] else ''
    print(f"  pos {p['position']}: tok{p['clean_top']}->tok{p['modified_top']}  "
          f"max_change={p['max_logit_change']:.4f}{status}")

In [ ]:
result = component_logit_lens_effect(model, tokens, layer=1)
print(f"Layer 1 component effects:")
print(f"  Attn changed {result['attn_changes']} predictions")
print(f"  MLP changed {result['mlp_changes']} predictions\n")
for p in result['per_position'][:5]:
    print(f"  pos {p['position']}: {p['pre_top']}->{p['post_attn_top']}->{p['post_mlp_top']}  "
          f"attn_change={p['attn_max_logit_change']:.4f}  mlp_change={p['mlp_max_logit_change']:.4f}")

In [ ]:
result = logit_lens_trajectory(model, tokens, position=-1, top_k=3)
print(f"Prediction trajectory at last position:\n")
for s in result['stages']:
    tops = ', '.join(f'tok{t["token"]}({t["prob"]:.3f})' for t in s['top_predictions'])
    print(f"  Layer {s['layer']}: [{tops}]  entropy={s['entropy']:.3f}")
print(f"\nFinal prediction: token {result['final_prediction']}")